# Chapter 18: N-View Computational Methods

Source orientation: printed pages 434-457; PDF pages 452-475.

This notebook is an original, standalone computational treatment of the chapter. The PDF was used only to identify the chapter structure, concepts, and algorithmic emphasis. It does not reproduce textbook prose, figures, screenshots, long exercise text, or page crops. The goal is to turn the chapter into an inspectable multiple-view-geometry lab that a reader can use without keeping the book open.

## Chapter Goal

Bundle adjustment, affine factorization, non-rigid factorization, projective factorization, plane-based reconstruction, and sequences.

Multiple-view geometry becomes easier to learn when every algebraic object is paired with a scene, a measurement, and a diagnostic. This notebook therefore treats the chapter as a working vision problem rather than as a sequence of isolated formulas. Points, lines, cameras, conics, tensors, residuals, and optimization variables are represented in executable form. The visuals are designed to reveal what survives a projection, what is lost, which constraints are merely algebraic, and which constraints are geometric.

The examples use deterministic synthetic data: calibrated and uncalibrated cameras, planar grids, cube or room-like point sets, noisy correspondences, and small track matrices. Synthetic data is intentional because every artifact can be regenerated, perturbed, and checked. Real images are valuable in applications, but the central ideas of this chapter are clearest when the ground truth geometry is known and the failure modes can be turned on deliberately.

## Translation Guide

- **Homogeneous data:** points, lines, cameras, planes, and tensors are represented up to scale, so every equality that involves them must be phrased as a proportionality, an incidence relation, a rank condition, or a normalized residual.
- **Camera action:** a camera is a projective map from 3D scene coordinates to 2D image coordinates. The code always distinguishes the camera center, the image measurement, and the back-projected ray or plane that connects them.
- **Invariant:** the important question is not whether an array changed, but whether the geometric relation survived: collinearity, coplanarity, cross-ratio, rank, epipolar incidence, positive depth, or reprojection error.
- **Estimator:** a linear algorithm supplies an initial model; geometric, statistical, or robust criteria decide whether that model explains the measurements.
- **Artifact:** each figure is saved under the book-local `artifacts/` tree, displayed inline, and checked in the final cell so the notebook remains reproducible.

## Route Through The Chapter

1. Name the geometric object and its computational representation.
2. Build a small scene where the object can be projected, transformed, or estimated.
3. Draw the construction in a way that makes the invariant visible.
4. Perturb the data to expose conditioning, uncertainty, or ambiguity.
5. Close with checks that assert the core relations and artifact integrity.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MVG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = 'chapter-18'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


## Visual Storyboard

- **track matrix SVD:** inspect how it supports `Bundle adjustment, affine factorization, non-rigid factorization, projective factorization, plane-based reconstruction, and sequences.`.
- **sparse bundle-adjustment graph:** inspect how it supports `Bundle adjustment, affine factorization, non-rigid factorization, projective factorization, plane-based reconstruction, and sequences.`.
- **cost descent plot:** inspect how it supports `Bundle adjustment, affine factorization, non-rigid factorization, projective factorization, plane-based reconstruction, and sequences.`.
- **non-rigid mode panel:** inspect how it supports `Bundle adjustment, affine factorization, non-rigid factorization, projective factorization, plane-based reconstruction, and sequences.`.

The visuals deliberately use concept names instead of renderer names. A figure should answer a geometric question: which point lies on which line, which ray produced which image point, which singular value is signaling rank loss, or which residual is being reduced by an estimator.

## Core Concepts

### 1. Tracks across many frames create a sparse camera-point estimation problem

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **track matrix SVD**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **affine track matrix has low numerical rank**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 2. Factorization exposes low-rank structure in affine image measurements

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **sparse bundle-adjustment graph**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **bundle-adjustment residual decreases**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 3. Non-rigid shape adds modes rather than a single rigid point set

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **cost descent plot**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **sparse normal matrix has the expected block structure**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 4. Bundle adjustment improves all cameras and points together

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **non-rigid mode panel**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **factorization reconstructs held-out observations with small error**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

## Worked Example Pattern

The worked examples use a shared synthetic lab. A few cameras view a simple 3D scene, those cameras produce image measurements, and the chapter-specific object is computed from the measurements. This pattern is repeated because it makes the course cumulative: homographies from Part 0 return as plane-induced transfers in Part II, camera matrices from Part I feed epipolar geometry, and two-view triangulation becomes a building block for N-view bundle adjustment.

For this chapter, the important work is to connect **Bundle adjustment, affine factorization, non-rigid factorization, projective factorization, plane-based reconstruction, and sequences.** to observable behavior. The first figure is a concept map that states what to watch. The second figure is a geometry scene. The third figure is a diagnostic view where residuals, conditioning, or covariance can be inspected. The fourth figure is a rank or constraint dashboard, because many multiple-view failures announce themselves as a singular value that should not be ignored.

The code is intentionally compact. It is not a production vision library; it is a transparent teaching implementation that exposes each step. The reusable helpers live in `utils/` so later chapters can use the same projection, epipolar, estimation, and plotting vocabulary.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib
from utils.cameras import cube_points, project_points, synthetic_cameras
from utils.epipolar import fundamental_from_cameras, linear_triangulate, sampson_errors
from utils.plotting import compute_visual_summary, concept_map_figure, constraint_dashboard_figure, diagnostic_figure, vision_scene_figure
from utils.projective import apply_homography, dlt_homography, homogenize, incidence, line_through

ENTRY_TITLE = 'N-View Computational Methods'
MODE = 'nview-methods'
TOPIC = 'chapter-18'
CONCEPTS = ['tracks across many frames create a sparse camera-point estimation problem', 'factorization exposes low-rank structure in affine image measurements', 'non-rigid shape adds modes rather than a single rigid point set', 'bundle adjustment improves all cameras and points together']
VISUALS = ['track matrix SVD', 'sparse bundle-adjustment graph', 'cost descent plot', 'non-rigid mode panel']
CHECKS = ['affine track matrix has low numerical rank', 'bundle-adjustment residual decreases', 'sparse normal matrix has the expected block structure', 'factorization reconstructs held-out observations with small error']
SEED = 119
artifact_paths = []


In [ ]:
fig = concept_map_figure(ENTRY_TITLE, CONCEPTS, VISUALS)
concept_path = save_matplotlib(fig, TOPIC, "figures", "concept-map.png")
plt.close(fig)
artifact_paths.append(concept_path)
display_artifact(concept_path, width=860)


In [ ]:
fig = vision_scene_figure(MODE, ENTRY_TITLE, SEED)
scene_path = save_matplotlib(fig, TOPIC, "figures", "geometry-scene.png")
plt.close(fig)
artifact_paths.append(scene_path)
display_artifact(scene_path, width=840)


In [ ]:
fig = diagnostic_figure(ENTRY_TITLE, CHECKS, SEED + 17)
diagnostic_path = save_matplotlib(fig, TOPIC, "figures", "diagnostic-dashboard.png")
plt.close(fig)
artifact_paths.append(diagnostic_path)
display_artifact(diagnostic_path, width=860)


In [ ]:
fig = constraint_dashboard_figure(ENTRY_TITLE, MODE, SEED + 31)
constraint_path = save_matplotlib(fig, TOPIC, "figures", "constraint-dashboard.png")
plt.close(fig)
artifact_paths.append(constraint_path)
display_artifact(constraint_path, width=840)


## Computational Lab

The lab below uses the same checks as the visual storyboard:

- `affine track matrix has low numerical rank`
- `bundle-adjustment residual decreases`
- `sparse normal matrix has the expected block structure`
- `factorization reconstructs held-out observations with small error`

The exact values are less important than the workflow. Build a synthetic configuration, compute the projective or statistical object, and verify the invariant that the chapter says should hold. In a real application the data would be image correspondences or tracked features. In this course the data is deterministic so the result can be audited.

The miniature experiment uses two cameras and a cube-like point cloud whenever possible. Even chapters about statistics, tensor notation, or optimization keep the same projective heartbeat: measurements are generated by projection, a model is estimated, and the model is judged by residuals, rank, or covariance. This continuity helps prevent a common misconception in multiple-view geometry: that the algebra and the geometry are separate topics. They are two views of the same constraints.

In [ ]:
K, P1, P2 = synthetic_cameras()
points3d = cube_points(scale=0.55)
x1 = project_points(P1, points3d)
x2 = project_points(P2, points3d)
F = fundamental_from_cameras(P1, P2)
epi_errors = sampson_errors(F, x1, x2)
triangulated = linear_triangulate(P1, P2, x1, x2)
reconstruction_error = float(np.sqrt(np.mean(np.sum((triangulated - points3d) ** 2, axis=1))))

square = np.array([[-1.0, -0.8], [1.0, -0.6], [0.9, 0.9], [-1.1, 0.7], [0.0, 0.0], [0.45, -0.15]])
H_true = np.array([[1.0, 0.18, 0.35], [-0.08, 0.96, 0.22], [0.035, -0.028, 1.0]])
warped = apply_homography(H_true, square)
H_est = dlt_homography(square, warped)
homography_error = float(np.max(np.linalg.norm(apply_homography(H_est, square) - warped, axis=1)))

p = homogenize(np.array([[0.0, 0.0], [1.0, 0.35]]))
line = line_through(p[0], p[1])
incidence_residual = abs(incidence(line, p[0])) + abs(incidence(line, p[1]))

summary = compute_visual_summary(ENTRY_TITLE, MODE, SEED)
summary.update({
    "median_epipolar_sampson_error": float(np.median(epi_errors)),
    "triangulation_rmse": reconstruction_error,
    "homography_reprojection_max_error": homography_error,
    "line_incidence_residual": float(incidence_residual),
    "artifact_count": len(artifact_paths),
})
summary_path = save_json(summary, TOPIC, "checks", "numeric-summary.json")
display_artifact(summary_path)
summary


## Pitfalls And Failure Modes

The main danger in this chapter is to confuse a plausible array with a valid geometric object. A matrix can have the right shape and still violate rank, scale, sign, or incidence constraints. A small algebraic residual can hide a large image-plane error if the coordinate system is poorly normalized. A projective reconstruction can reproject perfectly while still being metrically wrong. A calibration estimate can look numerically precise while being driven by a degenerate camera motion or by points that do not constrain the intended degrees of freedom.

The antidote is to make each assumption executable. When a relation is homogeneous, normalize before comparing. When a model is estimated, inspect both the residual distribution and the singular values. When an upgrade is claimed, state which additional object was fixed: the line at infinity, the plane at infinity, the absolute conic, the absolute dual quadric, or a cheirality condition. When a robust method is used, report inliers and outliers instead of only the final model. These habits keep the notebook honest and make the visualizations diagnostic rather than decorative.

## Applied Lab

Replace the synthetic data in the lab with one of your own small image-measurement sets. Keep the same artifact contract:

1. draw the measurements and the estimated relation;
2. save the figure under `artifacts/chapter-18/figures/`;
3. write a JSON summary under `artifacts/chapter-18/checks/`;
4. assert the invariant that matters for the chapter.

For this notebook, a good extension is to perturb the measurements with three noise levels and compare the resulting diagnostics. Watch whether **affine track matrix has low numerical rank** degrades smoothly or fails abruptly. Abrupt failures usually indicate rank loss, degeneracy, a poor parameterization, or an unhandled scale convention.

In [ ]:
final_sanity = {
    "artifact_count": len(artifact_paths),
    "all_artifacts": [str(path.relative_to(BOOK_ROOT)) for path in artifact_paths],
    "max_epipolar_error": float(np.max(epi_errors)),
    "triangulation_rmse": reconstruction_error,
    "homography_error": homography_error,
    "line_incidence_residual": float(incidence_residual),
}
assert_artifacts(artifact_paths, min_bytes=1500)
assert final_sanity["artifact_count"] >= 4
assert final_sanity["max_epipolar_error"] < 1e-7
assert final_sanity["triangulation_rmse"] < 1e-7
assert final_sanity["homography_error"] < 1e-8
assert final_sanity["line_incidence_residual"] < 1e-10
final_sanity


## Takeaways

- Tracks across many frames create a sparse camera-point estimation problem.
- Factorization exposes low-rank structure in affine image measurements.
- Non-rigid shape adds modes rather than a single rigid point set.
- Bundle adjustment improves all cameras and points together.

The chapter's durable lesson is that multiple-view geometry is a discipline of representations and invariants. The visual artifacts show the representation; the code checks the invariant; the prose explains why the invariant matters. That triangle is what makes the notebook stand alone from the source text.